# 23 — Party Transition Diagnostics v1

This notebook diagnoses whether the current watchlists are over-weighting Labour/Reform/Independent-facing terrain and under-detecting Conservative-adjacent terrain.

It does not recommend target selections. It creates diagnostic lists for political interpretation:

- Conservative Legacy / Transition Terrain
- Labour Stronghold Breakthrough Terrain
- Reform / Independent Disruption Terrain
- Green / Liberal Democrat Non-Core Terrain
- council-level party-transition summaries

The diagnostic is based on current/latest election shares unless historical ward-year party transitions are available.

In [13]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
DICTIONARY_DIR = DATA_DIR / "dictionaries"

for d in [PROCESSED_DIR, GEOGRAPHY_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed


In [14]:
INPUT_DIR = PROCESSED_DIR / "target_review_pack_v1_revised_caveats"
OUTPUT_DIR = PROCESSED_DIR / "party_transition_diagnostics_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

Input: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1


In [15]:
def find_file(filename, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = [
            PROCESSED_DIR / "target_review_pack_v1",
            PROCESSED_DIR / "target_model_v2",
            PROCESSED_DIR / "target_model_v1",
            PROCESSED_DIR / "report_assets_v1",
            PROCESSED_DIR / "election_results",
            PROCESSED_DIR,
            DATA_DIR / "raw" / "election_results",
            DATA_DIR / "raw",
            PROJECT_DIR,
            Path.cwd(),
        ]
    for folder in search_dirs:
        path = folder / filename
        if path.exists():
            return path
    # recursive fallback under processed and data/raw
    for root in [PROCESSED_DIR, DATA_DIR / "raw"]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional file missing:", filename)
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def save_csv(df, path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path, df.shape)
    return path


def boolish(s):
    return s.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def safe_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

## 23.1 Load revised review file

In [16]:
nw, _ = read_csv("north_west_revised_consolidated_review_v1", required=False)
if nw is None:
    # fallback to exact filename via folder
    path = INPUT_DIR / "north_west_revised_consolidated_review_v1.csv"
    if path.exists():
        nw = pd.read_csv(path, low_memory=False)
        print("Loaded fallback:", path, nw.shape)
    else:
        nw, _ = read_csv("north_west_consolidated_target_review_v1.csv")

print(nw.shape)
display(nw.head())

Optional file missing: north_west_revised_consolidated_review_v1
Loaded fallback: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_revised_consolidated_review_v1.csv (825, 59)
(825, 59)


,LAD25CD,LAD25NM,WD25CD,WD25NM,analysis_region,strategic_lane,strategic_lane_priority,strategic_lane_flags,is_clean_watchlist,is_caveated_watchlist,is_breakthrough_complacency,is_demographic_build,is_top100_watchlist,initial_watchlist_score,initial_watchlist_percentile,review_band,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,dominant_cluster_name,second_cluster_name,latest_election_source_year,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_margin_pct_allocated,has_major_caveat,boundary_caveat,county_election_caveat,data_confidence_note,model_review_summary,candidate_known,candidate_name,candidate_strength_rating,local_contact_known,member_presence,recent_sdp_activity,local_issue_hook,activist_accessibility,delivery_practicality,campaign_cost_estimate,manual_priority,manual_notes,reviewed_by,reviewed_date,technical_caveat_flag,boundary_caveat_flag,county_election_evidence_flag,missing_or_invalid_election_flag,sefton_boundary_flag,electoral_evidence_type,report_confidence_band,report_caveat_level,report_caveat_summary,revised_strategic_lane,include_in_main_report,include_in_serious_caveat_appendix
0,E07000121,Lancaster,E05014894,Heysham North,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,78.611324,99.617010,Review A,Review A Clean,81.165382,83.218786,62.951216,54.253246,95,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,2023.0,lab,independent,0.056387,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Sett...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,missing_or_invalid,Serious caveat / manual review,serious,missing/invalid election metrics,Manual Review / Serious Caveat,False,True
1,E06000009,Blackpool,E05015206,Waterloo,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,75.953840,98.930269,Review A,Review A Clean,82.480342,81.775542,52.212231,50.343514,95,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,2023.0,lab,con,0.031357,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Post...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,missing_or_invalid,Serious caveat / manual review,serious,missing/invalid election metrics,Manual Review / Serious Caveat,False,True
2,E08000004,Oldham,E05014652,Failsworth East,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,75.034628,98.335975,Review A,Review A Clean,70.661396,81.132744,63.853265,49.322090,100,Settled Working Families / Skilled Trades Suburbs,Rooted Older Homeowners,2024.0,lab,other,0.008601,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Sett...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,missing_or_invalid,Serious caveat / manual review,serious,missing/invalid election metrics,Manual Review / Serious Caveat,False,True
3,E08000001,Bolton,E05014823,Farnworth South,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,74.769921,98.217116,Review A,Review A Clean,87.221810,66.303463,57.404996,0.000000,100,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,2024.0,other,lab,0.098498,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Post...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,missing_or_invalid,Serious caveat / manual review,serious,missing/invalid election metrics,Manual Review / Serious Caveat,False,True
4,E06000009,Blackpool,E05015188,Bloomfield,North West,Clean Opportunity,1,Clean Opportunity; Breakthrough Build,True,False,True,False,True,73.799538,97.583201,Review A,Review A Clean,95.363090,69.017176,40.86921

## 23.2 Build transition scores

These are diagnostic scores, not final model scores. They help answer: what kind of party-political terrain is each ward sitting in?

In [17]:
def build_transition_scores(df):
    df = df.copy()

    for col in [
        "latest_election_con_share", "latest_election_lab_share", "latest_election_ld_share", "latest_election_green_share",
        "latest_election_reform_ukip_brexit_share", "latest_election_independent_share", "latest_election_other_share",
        "demographic_relevance_score", "breakthrough_complacency_score", "political_openness_score", "electoral_opportunity_score",
        "rooted_older_homeowners_share", "settled_working_families_skilled_trades_suburbs_share", "post_industrial_estates_deprived_working_communities_share",
        "stable_suburban_professionals_share", "cosmopolitan_young_professional_core_share", "student_transient_youth_share",
    ]:
        if col in df.columns:
            df[col] = to_num(df[col]).fillna(0)

    con = to_num(safe_col(df, "latest_election_con_share", 0)).fillna(0)
    lab = to_num(safe_col(df, "latest_election_lab_share", 0)).fillna(0)
    ld = to_num(safe_col(df, "latest_election_ld_share", 0)).fillna(0)
    green = to_num(safe_col(df, "latest_election_green_share", 0)).fillna(0)
    reform = to_num(safe_col(df, "latest_election_reform_ukip_brexit_share", 0)).fillna(0)
    ind = to_num(safe_col(df, "latest_election_independent_share", 0)).fillna(0)
    other = to_num(safe_col(df, "latest_election_other_share", 0)).fillna(0)

    demo = to_num(safe_col(df, "demographic_relevance_score", 0)).fillna(0)
    breakthrough = to_num(safe_col(df, "breakthrough_complacency_score", 0)).fillna(0)
    openness = to_num(safe_col(df, "political_openness_score", 0)).fillna(0)
    opportunity = to_num(safe_col(df, "electoral_opportunity_score", 0)).fillna(0)

    rooted = to_num(safe_col(df, "rooted_older_homeowners_share", 0)).fillna(0) * 100
    settled = to_num(safe_col(df, "settled_working_families_skilled_trades_suburbs_share", 0)).fillna(0) * 100
    postind = to_num(safe_col(df, "post_industrial_estates_deprived_working_communities_share", 0)).fillna(0) * 100
    stable_suburb = to_num(safe_col(df, "stable_suburban_professionals_share", 0)).fillna(0) * 100
    cosmopolitan = to_num(safe_col(df, "cosmopolitan_young_professional_core_share", 0)).fillna(0) * 100
    student = to_num(safe_col(df, "student_transient_youth_share", 0)).fillna(0) * 100

    # Conservative transition: not simply Conservative-held. Looks for Con strength, Con-adjacent demographics, and political fracture.
    con_adjacent_demo = (rooted * 0.45 + settled * 0.35 + stable_suburb * 0.20).clip(0, 100)
    con_fracture = ((reform + ind + other) * 100).clip(0, 100)
    df["conservative_transition_score"] = (
        con * 100 * 0.35 + con_adjacent_demo * 0.30 + con_fracture * 0.25 + openness * 0.10
    ).clip(0, 100)

    # Labour stronghold breakthrough: high Labour share + demographic fit + breakthrough score.
    lab_strength = (lab * 100).clip(0, 100)
    df["labour_stronghold_breakthrough_score"] = (
        lab_strength * 0.30 + demo * 0.35 + breakthrough * 0.25 + postind * 0.10
    ).clip(0, 100)

    # Reform/Independent disruption.
    disruption = ((reform + ind + other) * 100).clip(0, 100)
    df["reform_independent_disruption_score"] = (
        disruption * 0.45 + openness * 0.25 + opportunity * 0.15 + demo * 0.15
    ).clip(0, 100)

    # Green/LD non-core terrain, for diagnostic exclusion/alternative route.
    green_ld = ((green + ld) * 100).clip(0, 100)
    noncore_demo = (cosmopolitan * 0.45 + student * 0.25 + stable_suburb * 0.30).clip(0, 100)
    df["green_ld_noncore_score"] = (green_ld * 0.55 + noncore_demo * 0.35 + openness * 0.10).clip(0, 100)

    def primary_transition(row):
        scores = {
            "Conservative Legacy / Transition": row["conservative_transition_score"],
            "Labour Stronghold Breakthrough": row["labour_stronghold_breakthrough_score"],
            "Reform / Independent Disruption": row["reform_independent_disruption_score"],
            "Green / Liberal Democrat Non-Core": row["green_ld_noncore_score"],
        }
        return max(scores, key=scores.get)

    df["primary_party_transition_diagnostic"] = df.apply(primary_transition, axis=1)
    df["primary_party_transition_score"] = df[[
        "conservative_transition_score", "labour_stronghold_breakthrough_score", "reform_independent_disruption_score", "green_ld_noncore_score"
    ]].max(axis=1)

    return df

transitions = build_transition_scores(nw)
display(transitions[["LAD25NM", "WD25NM", "primary_party_transition_diagnostic", "primary_party_transition_score"]].head())

,LAD25NM,WD25NM,primary_party_transition_diagnostic,primary_party_transition_score
0,Lancaster,Heysham North,Labour Stronghold Breakthrough,41.971195
1,Blackpool,Waterloo,Labour Stronghold Breakthrough,41.453998
2,Oldham,Failsworth East,Reform / Independent Disruption,38.732437
3,Bolton,Farnworth South,Reform / Independent Disruption,37.380040
4,Blackpool,Bloomfield,Labour Stronghold Breakthrough,50.517395


## 23.3 Save diagnostic lists

In [18]:
base_cols = [c for c in [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region", "revised_strategic_lane", "report_confidence_band",
    "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
    "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket", "dominant_cluster_name", "second_cluster_name",
    "conservative_transition_score", "labour_stronghold_breakthrough_score", "reform_independent_disruption_score", "green_ld_noncore_score",
    "primary_party_transition_diagnostic", "primary_party_transition_score"
] if c in transitions.columns]

save_csv(transitions[base_cols].sort_values("primary_party_transition_score", ascending=False), OUTPUT_DIR / "north_west_party_transition_diagnostics_all_v1.csv")

save_csv(transitions.sort_values("conservative_transition_score", ascending=False)[base_cols].head(150), OUTPUT_DIR / "north_west_conservative_transition_wards_v1.csv")
save_csv(transitions.sort_values("labour_stronghold_breakthrough_score", ascending=False)[base_cols].head(150), OUTPUT_DIR / "north_west_labour_stronghold_breakthrough_wards_v1.csv")
save_csv(transitions.sort_values("reform_independent_disruption_score", ascending=False)[base_cols].head(150), OUTPUT_DIR / "north_west_reform_independent_disruption_wards_v1.csv")
save_csv(transitions.sort_values("green_ld_noncore_score", ascending=False)[base_cols].head(150), OUTPUT_DIR / "north_west_green_libdem_noncore_wards_v1.csv")

summary = transitions.groupby(["LAD25CD", "LAD25NM", "primary_party_transition_diagnostic"], dropna=False).agg(
    wards=("WD25CD", "nunique"),
    mean_transition_score=("primary_party_transition_score", "mean"),
    mean_model_score=("initial_watchlist_score", "mean"),
).reset_index()

council_wide = transitions.groupby(["LAD25CD", "LAD25NM"], as_index=False).agg(
    wards=("WD25CD", "nunique"),
    mean_conservative_transition_score=("conservative_transition_score", "mean"),
    mean_labour_stronghold_breakthrough_score=("labour_stronghold_breakthrough_score", "mean"),
    mean_reform_independent_disruption_score=("reform_independent_disruption_score", "mean"),
    mean_green_ld_noncore_score=("green_ld_noncore_score", "mean"),
    top_model_score=("initial_watchlist_score", "max"),
)

save_csv(summary, OUTPUT_DIR / "party_transition_summary_by_council_and_type_v1.csv")
save_csv(council_wide, OUTPUT_DIR / "party_transition_summary_by_council_v1.csv")

map_cols = [c for c in ["WD25CD", "WD25NM", "LAD25NM", "primary_party_transition_diagnostic", "primary_party_transition_score", "conservative_transition_score", "labour_stronghold_breakthrough_score", "reform_independent_disruption_score", "green_ld_noncore_score"] if c in transitions.columns]
save_csv(transitions[map_cols], OUTPUT_DIR / "north_west_party_transition_map_ready_v1.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_party_transition_diagnostics_all_v1.csv (825, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_conservative_transition_wards_v1.csv (150, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_labour_stronghold_breakthrough_wards_v1.csv (150, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_reform_independent_disruption_wards_v1.csv (150, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\north_west_green_libdem_noncore_wards_v1.csv (150, 22)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\party_transition_diagnostics_v1\party_transition_summary_by_council_and_type_v1.csv (69, 6)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\process

WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/party_transition_diagnostics_v1/north_west_party_transition_map_ready_v1.csv')

## 23.4 Interpretation note

This notebook is a diagnostic tool. Do not treat “Conservative Transition”, “Labour Stronghold Breakthrough” or “Reform / Independent Disruption” as target decisions. Use them to check whether the model is failing to see one route into the SDP coalition.